In [1]:
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = Path("..") / "data"
CLEAN = (DATA_DIR / "clean_sales.parquet").as_posix()
print(Path.cwd())

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE TABLE clean_sales AS SELECT * FROM read_parquet('{CLEAN}')")
print(con.execute("SELECT COUNT(*) FROM clean_sales").fetchall()[0][0], "rows loaded")

VOID_INVOICES = ["541431", "C541433", "581483", "C581484"]
voids = ",".join(f"'{i}'" for i in VOID_INVOICES)

C:\Users\kanak\retail-customer-analytics\notebooks
1021272 rows loaded


In [2]:
for label, cond in [("all clean rows", "TRUE"),
                    ("without the 2 reversed orders", f"invoice NOT IN ({voids})")]:
    print(label)
    display(con.execute(f"""
    SELECT
        ROUND(SUM(line_value) FILTER (WHERE NOT is_return), 2)  AS gross_sales,
        ROUND(-SUM(line_value) FILTER (WHERE is_return), 2)     AS returns_value,
        ROUND(100 * -SUM(line_value) FILTER (WHERE is_return)
                  / SUM(line_value) FILTER (WHERE NOT is_return), 2) AS return_pct,
        COUNT(DISTINCT invoice) FILTER (WHERE NOT is_return)    AS sales_invoices,
        COUNT(DISTINCT invoice) FILTER (WHERE is_return)        AS return_invoices
    FROM clean_sales WHERE {cond}
    """).fetchdf())

all clean rows


,gross_sales,returns_value,return_pct,sales_invoices,return_invoices
0,"19,643,861.64","716,462.57",3.65,39516,7406


without the 2 reversed orders


,gross_sales,returns_value,return_pct,sales_invoices,return_invoices
0,"19,398,208.44","470,809.37",2.43,39514,7404


In [3]:
monthly = con.execute(f"""
SELECT CAST(DATE_TRUNC('month', invoice_ts) AS DATE) AS month,
       ROUND(SUM(line_value) FILTER (WHERE NOT is_return), 0)  AS gross_sales,
       ROUND(-SUM(line_value) FILTER (WHERE is_return), 0)     AS returns_value,
       ROUND(SUM(line_value), 0)                               AS net_sales,
       COUNT(DISTINCT invoice) FILTER (WHERE NOT is_return)    AS orders,
       ROUND(100 * -SUM(line_value) FILTER (WHERE is_return)
                 / SUM(line_value) FILTER (WHERE NOT is_return), 1) AS return_pct
FROM clean_sales
WHERE invoice NOT IN ({voids})
  AND invoice_ts < TIMESTAMP '2011-12-01'
GROUP BY 1 ORDER BY 1
""").fetchdf()
display(monthly)

,month,gross_sales,returns_value,net_sales,orders,return_pct
0,2009-12-01,"797,901.00","19,558.00","778,343.00",1666,2.50
1,2010-01-01,"612,297.00","7,466.00","604,832.00",1049,1.20
2,2010-02-01,"537,910.00","12,575.00","525,334.00",1189,2.30
3,2010-03-01,"761,706.00","9,604.00","752,102.00",1647,1.30
4,2010-04-01,"646,341.00","10,025.00","636,316.00",1435,1.60
5,2010-05-01,"643,492.00","37,193.00","606,299.00",1484,5.80
6,2010-06-01,"696,583.00","26,373.00","670,210.00",1612,3.80
7,2010-07-01,"632,994.00","17,133.00","615,862.00",1507,2.70
8,2010-08-01,"674,038.00","12,398.00","661,640.00",1402,1.80
9,2010-09-01,"869,243.00","27,686.00","841,557.00",1789,3.20


In [4]:
display(con.execute(f"""
SELECT stock_code, ANY_VALUE(description) AS description,
       SUM(-quantity) AS units_returned,
       ROUND(-SUM(line_value), 2) AS returns_value
FROM clean_sales
WHERE is_return AND invoice NOT IN ({voids})
GROUP BY stock_code ORDER BY returns_value DESC LIMIT 10
""").fetchdf())

display(con.execute(f"""
WITH c AS (
    SELECT customer_id,
           -SUM(line_value) FILTER (WHERE is_return)     AS ret,
           SUM(line_value) FILTER (WHERE NOT is_return)  AS sales
    FROM clean_sales
    WHERE customer_id IS NOT NULL AND invoice NOT IN ({voids})
    GROUP BY customer_id
)
SELECT COUNT(*)                                          AS customers,
       COUNT(*) FILTER (WHERE ret > 0)                   AS customers_with_returns,
       ROUND(100.0 * COUNT(*) FILTER (WHERE ret > 0) / COUNT(*), 1) AS pct_with_returns,
       ROUND(SUM(ret) FILTER (WHERE ret > 0), 2)         AS returns_value
FROM c
""").fetchdf())

,stock_code,description,units_returned,returns_value
0,22423,REGENCY CAKESTAND 3 TIER,"1,450.00","16,545.30"
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,"3,637.00","9,387.10"
2,71477,COLOUR GLASS. STAR T-LIGHT HOLDER,"2,574.00","7,164.12"
3,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,"3,159.00","6,614.37"
4,79323W,WHITE CHERRY LIGHTS,"1,088.00","6,421.10"
5,21843,RETRO SPOT CAKE STAND,587.00,"5,270.25"
6,84078A,SET/4 WHITE RETRO STORAGE CUBES,147.00,"5,187.65"
7,23113,PANTRY CHOPPING BOARD,946.00,"4,803.06"
8,48185,DOORMAT FAIRY CAKE,692.00,"4,678.68"
9,47566B,TEA TIME PARTY BUNTING,"1,635.00","4,490.50"


,customers,customers_with_returns,pct_with_returns,returns_value
0,5875,2443,41.60,"464,336.83"


In [5]:
display(con.execute("""
SELECT year(invoice_ts) AS year,
       ROUND(SUM(line_value) FILTER (WHERE NOT is_return), 2) AS gross_sales,
       ROUND(SUM(line_value), 2)                              AS net_sales,
       COUNT(DISTINCT invoice) FILTER (WHERE NOT is_return)   AS orders
FROM clean_sales
WHERE year(invoice_ts) IN (2010, 2011)
  AND (month(invoice_ts) IN (10, 11) OR (month(invoice_ts) = 12 AND day(invoice_ts) <= 8))
GROUP BY 1 ORDER BY 1
""").fetchdf())

,year,gross_sales,net_sales,orders
0,2010,"2,882,149.17","2,815,788.12",5687
1,2011,"2,971,826.51","2,899,695.06",5528


In [6]:
display(con.execute("""
SELECT ROUND(SUM(line_value), 2) AS net_sales_2010,
       ROUND(100.0 * SUM(line_value) FILTER (WHERE month(invoice_ts) >= 10) / SUM(line_value), 1) AS q4_share_pct
FROM clean_sales WHERE year(invoice_ts) = 2010
""").fetchdf())

,net_sales_2010,q4_share_pct
0,"9,135,828.32",35.30


In [7]:
monthly.to_csv(DATA_DIR / "monthly_summary.csv", index=False)
print("saved monthly_summary.csv")

saved monthly_summary.csv
